# 02 · Recolección de datos: web scraping 🔵

**Módulo 2 · Sesión 4** — Recolección, limpieza y EDA · **Contenido opcional**

## Objetivos

Muchos proyectos empiezan sin dataset: hay que ir a buscarlo. Este notebook cubre la
extracción de datos desde páginas web:

1. Cómo funciona una petición HTTP y qué devuelve.
2. Navegar la estructura de un HTML con BeautifulSoup.
3. Extraer una tabla y convertirla en `DataFrame`.
4. El atajo de `pandas.read_html` y cuándo no sirve.
5. Qué hacer cuando el contenido lo genera JavaScript.
6. **Ética y legalidad**: lo que se puede y no se puede hacer.

> **Sobre la fuente.** Trabajamos contra un archivo HTML **local**
> (`../datos/pagina-ejemplo.html`) en lugar de un sitio real. No es pereza: las páginas
> reales cambian su estructura sin avisar y un notebook docente que dependa de una web viva
> deja de funcionar en cualquier momento. El código es idéntico; solo cambia de dónde sale
> el HTML. En la sección 6 se muestra cómo apuntarlo a una URL real.

## Paquetes

`requests`, `beautifulsoup4`, `pandas` (y `lxml`, que usa `read_html` por dentro).

In [ ]:
from io import StringIO
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup

SEMILLA = 42

## 1. Cómo se pide una página

Una petición HTTP tiene un método (`GET` para leer), una URL y unas cabeceras. La respuesta
trae un **código de estado** y un cuerpo, que para una página web es HTML en texto plano.

| Código | Significa |
|---|---|
| 200 | Todo bien |
| 301 / 302 | Redirección |
| 403 | Prohibido (a menudo, bloqueo de bots) |
| 404 | No existe |
| 429 | Demasiadas peticiones — vas muy rápido |
| 5xx | Error del servidor |

Con `requests`, una petición real se vería así:

```python
respuesta = requests.get(url, headers={"User-Agent": "curso-ml-utp/1.0"}, timeout=30)
respuesta.raise_for_status()      # lanza excepción si el código no es 2xx
html = respuesta.text
```

**Nunca omitas `timeout`**: sin él, una petición puede quedarse colgada indefinidamente.

Nosotros leemos el HTML desde el disco, que produce exactamente el mismo texto.

In [ ]:
ruta_html = Path("../datos/pagina-ejemplo.html")
html = ruta_html.read_text(encoding="utf-8")

print(f"Origen: {ruta_html.name}  ·  {len(html):,} caracteres")
print("\nPrimeras líneas:")
print("\n".join(html.splitlines()[:10]))

## 2. Parsear el HTML

El HTML es texto con estructura de árbol: etiquetas anidadas, cada una con atributos. **No
se procesa con expresiones regulares** — es un error clásico y frágil. Para eso está un
parser.

In [ ]:
sopa = BeautifulSoup(html, "html.parser")

print("Título de la página:", sopa.title.text)
print("Encabezado principal:", sopa.h1.text)
print("\nSecciones (h2):")
for h2 in sopa.find_all("h2"):
    print("  -", h2.text)

> **Sobre el parser.** Usamos `html.parser`, que viene con Python. `lxml` es más rápido y
> `html5lib` más tolerante con HTML mal formado, pero ambos son dependencias adicionales.
> Para páginas bien formadas, el de la biblioteca estándar basta.

### Buscar elementos

Los dos métodos que se usan el 90 % del tiempo:

- `find(...)` — el primer elemento que coincide.
- `find_all(...)` — todos los que coinciden, como lista.

Se pueden filtrar por etiqueta, por `id`, por clase CSS o por cualquier atributo.

In [ ]:
# Por id.
resumen = sopa.find("div", id="resumen")
print("Bloque de resumen:", resumen.get_text(strip=True))

# Por clase CSS (ojo: el argumento se llama class_, porque 'class' es palabra reservada).
print("\nDatos sueltos con class='dato':")
for span in sopa.find_all("span", class_="dato"):
    print(f"  {span['data-clave']:14s} = {span.text}")

Fíjate en `span['data-clave']`: los atributos de una etiqueta se leen como si fuera un
diccionario. Es la forma habitual de sacar el `href` de un enlace, por ejemplo.

In [ ]:
print("Enlaces de la página:")
for enlace in sopa.find("ul", id="enlaces").find_all("a"):
    print(f"  {enlace['href']:28s} -> {enlace.text}")

## 3. Extraer una tabla, fila por fila

Es el caso más frecuente. Una tabla HTML tiene esta estructura:

```html
<table>
  <thead><tr><th>Columna A</th><th>Columna B</th></tr></thead>
  <tbody>
    <tr><td>valor 1</td><td>valor 2</td></tr>
  </tbody>
</table>
```

El procedimiento: localizar la tabla, leer los `<th>` como encabezados, y recorrer cada
`<tr>` del cuerpo leyendo sus `<td>`.

In [ ]:
def extraer_tabla(sopa, id_tabla):
    """Convierte una <table> del HTML en un DataFrame."""
    tabla = sopa.find("table", id=id_tabla)
    if tabla is None:
        raise ValueError(f"No se encontró la tabla con id='{id_tabla}'")

    encabezados = [th.get_text(strip=True) for th in tabla.find("thead").find_all("th")]

    filas = []
    for tr in tabla.find("tbody").find_all("tr"):
        celdas = [td.get_text(strip=True) for td in tr.find_all("td")]
        if len(celdas) == len(encabezados):   # descarta filas mal formadas
            filas.append(celdas)

    return pd.DataFrame(filas, columns=encabezados)


programas = extraer_tabla(sopa, "tabla-programas")
print(programas.to_string(index=False))

### Todo llega como texto

Este es el punto que más problemas causa. Fíjate en los tipos:

In [ ]:
print(programas.dtypes.to_string())

`Matriculados` es texto, no un número. Cualquier operación aritmética fallaría o —peor—
haría algo inesperado (`"162" + "98"` da `"16298"`). **Convertir tipos es parte obligatoria
del scraping**, no un detalle posterior.

In [ ]:
programas["Matriculados"] = pd.to_numeric(programas["Matriculados"])
programas["Duración (semestres)"] = pd.to_numeric(programas["Duración (semestres)"])

print(programas.dtypes.to_string())
print(f"\nTotal de matriculados: {programas['Matriculados'].sum()}")
print(f"Programa más grande: {programas.loc[programas['Matriculados'].idxmax(), 'Programa']}")

> **Truco útil.** `pd.to_numeric(serie, errors="coerce")` convierte lo que puede y pone
> `NaN` en lo que no. Es la forma de sobrevivir a una columna donde algunos valores son
> `"1.234"`, otros `"N/D"` y otros `"—"`.

Extraigamos la segunda tabla con la misma función.

In [ ]:
cohortes = extraer_tabla(sopa, "tabla-cohortes")
for columna in ["Admitidos", "Graduados", "Deserción (%)"]:
    cohortes[columna] = pd.to_numeric(cohortes[columna])

print(cohortes.to_string(index=False))
print(f"\nDeserción promedio: {cohortes['Deserción (%)'].mean():.1f}%")
print(f"Tendencia (primera vs. última cohorte): "
      f"{cohortes['Deserción (%)'].iloc[0]:.1f}% -> {cohortes['Deserción (%)'].iloc[-1]:.1f}%")

## 4. El atajo: `pandas.read_html`

Si lo único que quieres son las tablas de una página, pandas las extrae todas de una vez.

In [ ]:
tablas = pd.read_html(StringIO(html))

print(f"Tablas encontradas: {len(tablas)}\n")
for i, tabla in enumerate(tablas):
    print(f"--- Tabla {i}: {tabla.shape[0]} filas x {tabla.shape[1]} columnas ---")
    print(tabla.head(2).to_string(index=False))
    print()

Mucho más corto, y además **convierte los tipos automáticamente**:

In [ ]:
print(tablas[0].dtypes.to_string())

### ¿Cuándo usar cada uno?

| Usa `read_html` | Usa BeautifulSoup |
|---|---|
| Solo necesitas tablas `<table>` bien formadas | Los datos no están en una tabla |
| Quieres inferencia de tipos gratis | Necesitas atributos (`href`, `data-*`) |
| Prototipado rápido | La tabla está mal formada o es irregular |
| | Tienes que recorrer varias páginas siguiendo enlaces |

`read_html` requiere `lxml` o `html5lib` instalados. Si no los tienes, usa BeautifulSoup.

## 5. Cuando el contenido lo genera JavaScript

Muchos sitios modernos entregan un HTML casi vacío y cargan los datos después, con
JavaScript. Si `requests` devuelve una página sin los datos que ves en el navegador, es esto.

Tres salidas, en orden de preferencia:

1. **Busca la API.** Abre las herramientas de desarrollo del navegador (F12), pestaña
   *Network*, y mira qué peticiones hace la página. Casi siempre hay una que devuelve JSON
   limpio. Consumir esa API directamente es más rápido, más estable y más respetuoso que
   raspar el HTML.
2. **Busca datos abiertos.** Muchas instituciones publican los mismos datos en portales de
   datos abiertos o vía API oficial. Siempre es preferible.
3. **Automatiza un navegador** (Selenium, Playwright) como último recurso: es lento, frágil
   y pesado.

## 6. Ética y legalidad

El scraping tiene límites que no son opcionales.

### Antes de raspar

- **Revisa `robots.txt`** (`https://sitio.com/robots.txt`): indica qué rutas el sitio pide
  no rastrear. No es legalmente vinculante en todos los países, pero ignorarlo es actuar de
  mala fe.
- **Lee los términos de servicio.** Algunos sitios prohíben explícitamente la extracción
  automatizada.
- **Pregunta si existe una API oficial.** Casi siempre la respuesta es sí, y resuelve el
  problema mejor.

### Mientras raspas

- **Identifícate** con un `User-Agent` descriptivo y un contacto.
- **Espera entre peticiones** (`time.sleep(1)` como mínimo). Un bucle sin pausas es
  indistinguible de un ataque de denegación de servicio.
- **Guarda el HTML crudo** la primera vez. Si necesitas cambiar el parseo, trabajas sobre el
  archivo local en vez de volver a golpear el servidor.
- **Nunca extraigas datos personales** sin base legal. En Colombia aplica la Ley 1581 de
  2012 de protección de datos personales; en Europa, el RGPD.

Un ejemplo responsable, listo para adaptar:

```python
import time
from pathlib import Path

import requests

CABECERAS = {"User-Agent": "curso-ml-utp/1.0 (delram@utp.edu.co)"}
CACHE = Path("../datos/crudos")
CACHE.mkdir(parents=True, exist_ok=True)

def descargar(url, nombre):
    """Descarga una página, guardando una copia local para no repetir la petición."""
    destino = CACHE / nombre
    if destino.exists():                       # ya la tenemos: no molestamos al servidor
        return destino.read_text(encoding="utf-8")

    respuesta = requests.get(url, headers=CABECERAS, timeout=30)
    respuesta.raise_for_status()
    destino.write_text(respuesta.text, encoding="utf-8")
    time.sleep(1)                              # pausa entre peticiones
    return respuesta.text

html = descargar("https://ejemplo.edu.co/posgrados", "posgrados.html")
sopa = BeautifulSoup(html, "html.parser")
```

Con esa función, todo el código de las secciones 2 a 4 funciona igual contra un sitio real.

## 7. El scraping es solo el principio

Los datos raspados llegan **sucios por construcción**: todo es texto, hay espacios de más,
caracteres invisibles, formatos numéricos mezclados y valores como `"N/D"` o `"-"`.

In [ ]:
# Ejemplo de la clase de suciedad típica de una extracción real.
sucio = pd.DataFrame(
    {
        "programa": [" Sistemas ", "Estadística\n", "Eléctrica"],
        "matriculados": ["162", "1.098", "N/D"],
        "desercion": ["18,5%", "20.4%", "—"],
    }
)
print("Como llega:")
print(sucio.to_string(index=False))
print()
print(sucio.dtypes.to_string())

In [ ]:
limpio = sucio.copy()
limpio["programa"] = limpio["programa"].str.strip()
limpio["matriculados"] = pd.to_numeric(
    limpio["matriculados"].str.replace(".", "", regex=False), errors="coerce"
)
limpio["desercion"] = pd.to_numeric(
    limpio["desercion"].str.rstrip("%").str.replace(",", ".", regex=False), errors="coerce"
)

print("Después de limpiar:")
print(limpio.to_string(index=False))
print()
print(limpio.dtypes.to_string())
print(f"\nValores no convertibles marcados como NaN: {limpio.isna().sum().sum()}")

Fíjate en las decisiones: `"1.098"` usa el punto como separador de miles y `"18,5%"` la coma
como decimal — convenciones locales que hay que interpretar. `"N/D"` y `"—"` se convierten
en `NaN` gracias a `errors="coerce"`, y a partir de ahí son valores faltantes normales, con
todo lo que vimos en el notebook 01.

## Resumen

| Paso | Herramienta | Cuidado principal |
|---|---|---|
| Descargar | `requests.get(..., timeout=)` | Cabeceras, pausas, caché local |
| Parsear | `BeautifulSoup(html, "html.parser")` | Nunca con expresiones regulares |
| Localizar | `find` / `find_all` | `class_` en vez de `class` |
| Tablas | `pd.read_html` | Solo sirve para `<table>` bien formadas |
| Convertir | `pd.to_numeric(..., errors="coerce")` | **Todo llega como texto** |
| Respetar | `robots.txt`, términos, ley de datos | Preferir siempre la API oficial |

## Para practicar

1. Extrae de `pagina-ejemplo.html` la tasa de graduación (`Graduados / Admitidos`) por
   cohorte y ordena de mayor a menor.
2. Escribe una función que reciba la sopa y devuelva un diccionario con todos los `span` de
   clase `dato`, usando `data-clave` como llave.
3. Modifica `extraer_tabla` para que convierta automáticamente a numérica toda columna en la
   que **todos** los valores sean convertibles.
4. Abre `pagina-ejemplo.html` en un editor, añade una fila a la tabla de cohortes con un
   valor `"N/D"` en graduados, y comprueba que tu código lo maneja sin romperse.